In [3]:
import pandas as pd
import numpy as np
import pickle
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD

In [4]:
movies = pd.read_csv("../Data/movies.csv",low_memory=False)


In [5]:
ratings = pd.read_csv(
    "../Data/ratings.csv",
    usecols=["userId", "movieId", "rating"],  # skip timestamp — saves RAM
    dtype={
        "userId": "int32",      # saves RAM vs int64
        "movieId": "int32",
        "rating": "float32"
    }
)

In [6]:
movies.shape

(62423, 3)

In [7]:
ratings.shape

(25000095, 3)

In [8]:
ratings.head(10)

,userId,movieId,rating
0,1,296,5.0
1,1,306,3.5
2,1,307,5.0
3,1,665,5.0
4,1,899,3.5
5,1,1088,4.0
6,1,1175,3.5
7,1,1217,3.5
8,1,1237,5.0
9,1,1250,4.0


In [9]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [10]:
movie_counts  = ratings['movieId'].value_counts()


In [11]:
popular_movies = movie_counts[movie_counts >= 5000].index

In [12]:
ratings = ratings[ratings['movieId'].isin(popular_movies)]


In [13]:
ratings.shape

(16358020, 3)

In [14]:
user_counts  = ratings['userId'].value_counts()
active_users = user_counts[user_counts >= 500].index  # 100+ ratings
ratings = ratings[ratings['userId'].isin(active_users)]

In [15]:
ratings.shape

(1931334, 3)

In [16]:
print("Unique users:",  ratings['userId'].nunique())
print("Unique movies:", ratings['movieId'].nunique())

Unique users: 3024
Unique movies: 1223


In [17]:
user_movie_matrix = ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
).fillna(0)


In [18]:
user_movie_matrix.shape

(3024, 1223)

In [19]:
user_movie_matrix

movieId,1,2,3,5,6,7,10,11,14,16,...,134853,139385,142488,148626,152081,164179,166528,168250,168252,176371
userId,,,,,,,,,,,,,,,,,,,,,
187,3.5,3.5,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0
426,2.5,0.0,0.0,0.0,3.0,0.0,3.0,1.0,0.5,3.5,...,0.0,4.0,5.0,4.0,0.5,1.0,0.0,0.5,0.0,3.0
548,4.5,4.0,0.0,0.0,4.0,0.0,3.5,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
626,4.5,4.0,0.0,0.0,0.0,4.5,4.0,4.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
757,3.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,4.5,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
162271,4.0,1.0,0.0,0.0,3.5,3.0,0.0,4.0,4.5,4.5,...,3.0,2.5,4.0,2.5,2.5,3.5,0.0,4.0,2.5,2.5
162286,4.0,5.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,4.0,...,5.0,5.0,4.0,4.0,4.5,4.0,4.5,5.0,5.0,5.0
162349,5.0,1.0,0.0,0.0,5.0,0.0,5.0,0.0,0.0,5.0,...,5.0,5.0,5.0,0.0,5.0,5.0,5.0,5.0,5.0,5.0


In [20]:
sparse_matrix = csr_matrix(user_movie_matrix.values)

In [21]:
sparse_matrix 

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 1931334 stored elements and shape (3024, 1223)>

In [22]:
svd = TruncatedSVD(n_components=50, random_state=42)
svd.fit(sparse_matrix)

,n_components,50
,algorithm,'randomized'
,n_iter,5
,n_oversamples,10
,power_iteration_normalizer,'auto'
,random_state,42
,tol,0.0


In [23]:
movie_factors = svd.components_.T
print("Movie factors shape:", movie_factors.shape)


Movie factors shape: (1223, 50)


In [24]:
import os, pickle
os.makedirs("../models", exist_ok=True)

with open("../models/cf_movie_factors.pkl", "wb") as f:
    pickle.dump(movie_factors, f)

with open("../models/cf_movie_ids.pkl", "wb") as f:
    pickle.dump(list(user_movie_matrix.columns), f)

with open("../models/cf_movies_df.pkl", "wb") as f:
    pickle.dump(movies, f)

print("CF model saved!")
print(f"   Movies: {len(user_movie_matrix.columns)}")
print(f"   Users:  {len(user_movie_matrix.index)}")

CF model saved!
   Movies: 1223
   Users:  3024


In [44]:
# Load saved pickles
with open("../models/cf_movie_factors.pkl", "rb") as f:
    movie_factors = pickle.load(f)

with open("../models/cf_movie_ids.pkl", "rb") as f:
    movie_ids = pickle.load(f)

with open("../models/cf_movies_df.pkl", "rb") as f:
    movies_df = pickle.load(f)

print("Loaded CF model successfully!")
print("Movie factors shape:", movie_factors.shape)
print("Movie IDs count:", len(movie_ids))
print(movies_df.head(3))

Loaded CF model successfully!
Movie factors shape: (1223, 50)
Movie IDs count: 1223
   movieId                    title  \
0        1         Toy Story (1995)   
1        2           Jumanji (1995)   
2        3  Grumpier Old Men (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  


In [45]:
# Build movie_user_matrix for KNN (transpose of user_movie_matrix)
movie_user_matrix = user_movie_matrix.T
print("Movie-user matrix shape:", movie_user_matrix.shape)

Movie-user matrix shape: (1223, 3024)


In [56]:
# Test recommendation
def cf_recommend(query_title, top_n=10):
    match = movies_df[movies_df["title"].str.lower().str.contains(query_title.lower(),regex=False)]
    if match.empty:
        return "Movie not found"
    
    movie_id = match.iloc[0]["movieId"]
    
    if movie_id not in movie_ids:
        return "Movie not in CF dataset"
    
    idx = movie_ids.index(movie_id)
    query_vector = movie_factors[idx]
    
    norms = np.linalg.norm(movie_factors, axis=1)
    sims  = movie_factors @ query_vector / (norms * np.linalg.norm(query_vector) + 1e-9)
    
    order = np.argsort(-sims)
    
    results = []
    for i in order:                        
        if i == idx:
            continue
        mid = movie_ids[i]
        title_row = movies_df[movies_df["movieId"] == mid]
        if title_row.empty:
            continue
        results.append((title_row.iloc[0]["title"], round(float(sims[i]), 4)))
        if len(results) >= top_n:
            break
    return results
    

In [57]:
from sklearn.neighbors import NearestNeighbors
import time
knn_sparse = csr_matrix(movie_user_matrix.values)
knn = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=11)
knn.fit(knn_sparse)
print("KNN model trained!")


KNN model trained!


In [58]:
def knn_recommend(query_title, top_n=10):
    match = movies_df[movies_df["title"].str.lower().str.contains(query_title.lower(),regex=False)]
    if match.empty:
        return []
    
    movie_id = match.iloc[0]["movieId"]
    if movie_id not in movie_user_matrix.index:
        return []
    
    idx = movie_user_matrix.index.get_loc(movie_id)
    distances, indices = knn.kneighbors(
        movie_user_matrix.iloc[idx].values.reshape(1, -1),
        n_neighbors=top_n + 1
    )
    
    results = []
    for i, dist in zip(indices[0][1:], distances[0][1:]):
        mid = movie_user_matrix.index[i]
        title_row = movies_df[movies_df["movieId"] == mid]
        if not title_row.empty:
            results.append((title_row.iloc[0]["title"], round(1 - dist, 4)))
    return results

# Quick test
print("SVD:", cf_recommend("Toy Story", top_n=5))
print("KNN:", knn_recommend("Toy Story", top_n=5))

SVD: [('Toy Story 2 (1999)', 0.8748), ("Bug's Life, A (1998)", 0.7133), ('Monsters, Inc. (2001)', 0.7051), ('Finding Nemo (2003)', 0.6758), ('Lion King, The (1994)', 0.6673)]
KNN: [('Jurassic Park (1993)', np.float32(0.938)), ('Back to the Future (1985)', np.float32(0.9371)), ('Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981)', np.float32(0.9348)), ('Matrix, The (1999)', np.float32(0.9314)), ('Star Wars: Episode V - The Empire Strikes Back (1980)', np.float32(0.9308))]


In [59]:
#compariso result quality
test_movies = ["Toy Story", "Matrix", "Inception"]

for movie in test_movies:
    print(f"{chr(61)*50}")
    print(f"Movie: {movie}")
    print(f"{chr(61)*50}")
    
    svd_recs = cf_recommend(movie, top_n=5)
    knn_recs = knn_recommend(movie, top_n=5)
    
    print("SVD recommendations:")
    if isinstance(svd_recs, list):
        for title, score in svd_recs:
            print(f"   {title:40s} score: {score}")
    else:
        print(f"   {svd_recs}")
    
    print("KNN recommendations:")
    for title, score in knn_recs:
        print(f"   {title:40s} similarity: {score}")


Movie: Toy Story
SVD recommendations:
   Toy Story 2 (1999)                       score: 0.8748
   Bug's Life, A (1998)                     score: 0.7133
   Monsters, Inc. (2001)                    score: 0.7051
   Finding Nemo (2003)                      score: 0.6758
   Lion King, The (1994)                    score: 0.6673
KNN recommendations:
   Jurassic Park (1993)                     similarity: 0.9380000233650208
   Back to the Future (1985)                similarity: 0.9370999932289124
   Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981) similarity: 0.9348000288009644
   Matrix, The (1999)                       similarity: 0.9314000010490417
   Star Wars: Episode V - The Empire Strikes Back (1980) similarity: 0.9308000206947327
Movie: Matrix
SVD recommendations:
   Sixth Sense, The (1999)                  score: 0.7626
   Fight Club (1999)                        score: 0.7569
   Men in Black (a.k.a. MIB) (1997)         score: 0.7421
   Twelve Monkey

In [60]:
# COMPARISON 2: Speed Test
print(f"{chr(61)*50}")
print("SPEED COMPARISON (100 requests each)")
print(f"{chr(61)*50}")

start = time.time()
for _ in range(100):
    cf_recommend("Toy Story", top_n=10)
svd_time = (time.time() - start) / 100
print(f"SVD avg: {svd_time*1000:.2f} ms")

start = time.time()
for _ in range(100):
    knn_recommend("Toy Story", top_n=10)
knn_time = (time.time() - start) / 100
print(f"KNN avg: {knn_time*1000:.2f} ms")
print(f"SVD is {knn_time/svd_time:.1f}x faster than KNN")



SPEED COMPARISON (100 requests each)
SVD avg: 17.04 ms
KNN avg: 30.63 ms
SVD is 1.8x faster than KNN


In [61]:
print(f"\n{'='*50}")
print("COVERAGE (out of 50 random movies)")
print(f"{'='*50}")

# Only sample from movies actually in CF model
valid_movie_ids = set(movie_ids)
valid_movies = movies_df[movies_df["movieId"].isin(valid_movie_ids)]
sample = valid_movies["title"].sample(50, random_state=42)

svd_cov = sum(1 for t in sample if isinstance(cf_recommend(t, top_n=1), list) and len(cf_recommend(t, top_n=1)) > 0)
knn_cov = sum(1 for t in sample if len(knn_recommend(t, top_n=1)) > 0)

print(f"SVD coverage: {svd_cov}/50")
print(f"KNN coverage: {knn_cov}/50")


COVERAGE (out of 50 random movies)
SVD coverage: 50/50
KNN coverage: 50/50
